# CIFAR-10 MixUp & CutMix Augmentation

**Author:** Irida

## What This Notebook Does

This notebook **improves** the baseline project by replacing the simple Cutout augmentation with two more powerful techniques:

- **MixUp**: Blends two images and their labels together (e.g., 60% cat + 40% dog → model learns soft boundaries)
- **CutMix**: Cuts a patch from one image and pastes it onto another, mixing labels proportionally


### Why MixUp & CutMix are better than Cutout
| Technique | What it does | Limitation |
|---|---|---|
| Cutout (baseline) | Blacks out a random square | Loses information — the patch is wasted |
| **MixUp** (ours) | Blends two images linearly | Uses ALL information from two samples |
| **CutMix** (ours) | Pastes a real patch from another image | Replaces the black square with real pixels |

## 0. Environment Setup (Google Colab + GPU)

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import time
from pathlib import Path

# Detect Colab and adjust working directory to project root
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def adjust_working_dir():
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / 'Dataset' / 'batches.meta.mat').exists():
            os.chdir(p)
            print(f"Set working directory to project root: {p}")
            return p
    if IN_COLAB:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_path = Path('/content/drive/MyDrive')
        if drive_path.exists():
            for root, dirs, files in os.walk(drive_path):
                if 'batches.meta.mat' in files and 'Dataset' in root:
                    p = Path(root).parent
                    os.chdir(p)
                    print(f"Found and set working directory to Colab Drive project root: {p}")
                    return p
    print(f"Warning: Project root not found. Current dir: {cwd}")
    return cwd

project_root = adjust_working_dir()

# Set up paths relative to TensorFlow directory under project root
BASE_PATH = Path('TensorFlow')
model_dir = BASE_PATH / 'Model'
figure_dir = BASE_PATH / 'Figure'
preprocessing_dir = BASE_PATH / 'Preprocessing'

os.makedirs(model_dir, exist_ok=True)
os.makedirs(figure_dir, exist_ok=True)
os.makedirs(preprocessing_dir, exist_ok=True)

print(f'Models  → {model_dir}')
print(f'Figures → {figure_dir}')

In [ ]:
import tensorflow as tf
print(f'TensorFlow version : {tf.__version__}')
print(f'GPUs available     : {tf.config.list_physical_devices("GPU")}')
print(f'Built with CUDA    : {tf.test.is_built_with_cuda()}')

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

print('All imports successful.')

## 2. Load Preprocessed Data

We load the `.npz` file from preprocessing notebook.
This means we **reuse** the existing cleaned data (denoised + CLAHE + sharpened) and apply our **new augmentation on top**.

In [ ]:
from pathlib import Path
import numpy as np
import os

# Ensure BASE_PATH is accessible and define preprocessing_dir consistently.
preprocessing_data_path = BASE_PATH / 'Preprocessing'

def find_file_in_repo(fname):
    """Searches for a file across current directory, parent trees, and Google Drive."""
    cwd = Path.cwd().resolve()
    drive_base = Path('/content/drive/MyDrive')

    # Define fallback search combinations including the derived preprocessing_data_path
    search_paths = [cwd, *cwd.parents, drive_base, preprocessing_data_path]
    subdirs = [Path(''), Path('Preprocessing'), Path('Dataset')]

    # 1. Standard check in all defined search paths and their subdirs
    for root in search_paths:
        for subdir in subdirs:
            candidate = root / subdir / fname
            if candidate.exists():
                return candidate

    # Removed the previous specific 'drive_preproc' fallback as it was hardcoded and inconsistent.

    raise FileNotFoundError(f"Could not find {fname} starting from {cwd} or within Google Drive (checked directories include: {preprocessing_data_path}).")

# 1. Locate and load the dataset
npz_path = find_file_in_repo('cifar10_preprocessed.npz')
print(f"Loading preprocessed data from: {npz_path}")

with np.load(npz_path) as data:
    # 2. Extract and cast image arrays
    X_train = data['X_train'].astype(np.float32)
    X_val   = data['X_val'].astype(np.float32)
    X_test  = data['X_test'].astype(np.float32)

    # 3. Precise fallback check for labels using a clean helper
    def get_labels(split):
        for key in [f'y_{split}_int', f'y_{split}']:
            if key in data:
                return data[key].astype(np.int32)
        raise KeyError(f"Label array for '{split}' split not found in npz file.")

    y_train = get_labels('train')
    y_val   = get_labels('val')
    y_test  = get_labels('test')

# Metadata definitions
NUM_CLASSES = 10
LABEL_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

print(f"Loaded: X_train={X_train.shape}, X_val={X_val.shape}, X_test={X_test.shape}")
print(f"Labels: y_train={y_train.shape}, y_val={y_val.shape}, y_test={y_test.shape}")

## 3. MixUp & CutMix Augmentation

### 3.1 What is MixUp?

MixUp linearly interpolates between two training examples:

```
mixed_image = λ * image_A + (1 - λ) * image_B
mixed_label = λ * label_A + (1 - λ) * label_B
```

Where λ is sampled from a **Beta distribution** (Beta(α, α), typically α=0.2).

**Why it works:** The model is forced to learn linear relationships between classes, which makes the decision boundaries smoother and reduces overconfidence.

### 3.2 What is CutMix?

CutMix cuts a rectangular patch from image B and pastes it into image A:

```
mixed_image = image_A with a patch replaced by image_B
mixed_label = λ * label_A + (1 - λ) * label_B
```

Where λ is proportional to the **area** of the patch.

**Why it works:** Unlike Cutout (which wastes information with a black square), CutMix fills that region with real pixels from another image — no information is wasted.

In [ ]:
# ─────────────────────────────────────────────
#  MixUp
# ─────────────────────────────────────────────
def mixup(images, labels, alpha=0.2):
    """Apply MixUp augmentation to a batch.

    Args:
        images : float32 tensor of shape (batch, H, W, C)
        labels : int32 tensor of shape (batch,)
        alpha  : Beta distribution parameter (higher = more mixing)
    Returns:
        mixed_images : blended images
        mixed_labels : soft one-hot labels
    """
    batch_size = tf.shape(images)[0]

    # Sample lambda from Beta(alpha, alpha)
    lam = tf.cast(
        tf.random.stateless_parameterized_truncated_normal(
            shape=[batch_size], means=0.5, stddevs=0.2,
            minvals=0.0, maxvals=1.0, seed=[SEED, 0]
        ), tf.float32
    )
    # Use numpy Beta sampling (simpler and reliable)
    lam_np = np.random.beta(alpha, alpha, batch_size).astype(np.float32)
    lam_np = np.maximum(lam_np, 1 - lam_np)   # always keep dominant image
    lam = tf.constant(lam_np)

    # Shuffle indices for pairing
    indices = tf.random.shuffle(tf.range(batch_size))
    images2 = tf.gather(images, indices)
    labels2 = tf.gather(labels, indices)

    # Mix images
    lam_img = tf.reshape(lam, [batch_size, 1, 1, 1])
    mixed_images = lam_img * images + (1 - lam_img) * images2

    # Mix labels (soft one-hot)
    labels_oh  = tf.one_hot(labels,  NUM_CLASSES)
    labels2_oh = tf.one_hot(labels2, NUM_CLASSES)
    lam_label  = tf.reshape(lam, [batch_size, 1])
    mixed_labels = lam_label * labels_oh + (1 - lam_label) * labels2_oh

    return mixed_images, mixed_labels


# ─────────────────────────────────────────────
#  CutMix
# ─────────────────────────────────────────────
def cutmix(images, labels, alpha=1.0):
    """Apply CutMix augmentation to a batch.

    Args:
        images : float32 tensor of shape (batch, H, W, C)
        labels : int32 tensor of shape (batch,)
        alpha  : Beta distribution parameter for patch size
    Returns:
        mixed_images : images with pasted patches
        mixed_labels : soft one-hot labels (proportional to patch area)
    """
    batch_size = tf.shape(images)[0]
    H = tf.shape(images)[1]
    W = tf.shape(images)[2]

    # Sample lambda (ratio of patch area)
    lam_np = np.random.beta(alpha, alpha, 1)[0]
    lam = float(lam_np)

    # Compute patch size
    cut_ratio = np.sqrt(1.0 - lam)
    cut_h = int(32 * cut_ratio)
    cut_w = int(32 * cut_ratio)

    # Random center
    cx = np.random.randint(32)
    cy = np.random.randint(32)

    x1 = max(cx - cut_w // 2, 0)
    y1 = max(cy - cut_h // 2, 0)
    x2 = min(cx + cut_w // 2, 32)
    y2 = min(cy + cut_h // 2, 32)

    # Actual lambda based on real patch area
    lam = 1 - ((x2 - x1) * (y2 - y1)) / (32 * 32)

    # Shuffle indices for pairing
    indices = tf.random.shuffle(tf.range(batch_size))
    images2 = tf.gather(images, indices)
    labels2 = tf.gather(labels, indices)

    # Build mixed images using numpy for patch replacement
    images_np  = images.numpy()
    images2_np = images2.numpy()
    mixed = images_np.copy()
    mixed[:, y1:y2, x1:x2, :] = images2_np[:, y1:y2, x1:x2, :]
    mixed_images = tf.constant(mixed)

    # Mix labels proportional to patch area
    labels_oh  = tf.one_hot(labels,  NUM_CLASSES)
    labels2_oh = tf.one_hot(labels2, NUM_CLASSES)
    mixed_labels = lam * labels_oh + (1 - lam) * labels2_oh

    return mixed_images, mixed_labels

print('MixUp and CutMix functions defined.')

## 4. Visualize MixUp & CutMix

Let's visually confirm the augmentations work correctly before training.

In [ ]:
# Take a small batch for visualization
sample_imgs   = tf.constant(X_train[:16])
sample_labels = tf.constant(y_train[:16])

mixup_imgs,  mixup_labels  = mixup(sample_imgs, sample_labels)
cutmix_imgs, cutmix_labels = cutmix(sample_imgs, sample_labels)

fig, axes = plt.subplots(3, 8, figsize=(18, 7))
fig.suptitle('Original → MixUp → CutMix', fontsize=14, fontweight='bold')

for i in range(8):
    # Original
    axes[0, i].imshow(np.clip(X_train[i], 0, 1))
    axes[0, i].set_title(label_names[y_train[i]], fontsize=8)
    axes[0, i].axis('off')

    # MixUp
    axes[1, i].imshow(np.clip(mixup_imgs[i].numpy(), 0, 1))
    top_cls = np.argmax(mixup_labels[i].numpy())
    axes[1, i].set_title(f'mix:{label_names[top_cls]}', fontsize=7)
    axes[1, i].axis('off')

    # CutMix
    axes[2, i].imshow(np.clip(cutmix_imgs[i].numpy(), 0, 1))
    top_cls = np.argmax(cutmix_labels[i].numpy())
    axes[2, i].set_title(f'cut:{label_names[top_cls]}', fontsize=7)
    axes[2, i].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(figure_dir, 'mixup_cutmix_samples.png'), dpi=150)
plt.show()
print('Visualization saved.')

## 5. Build tf.data Pipeline with MixUp & CutMix

We apply MixUp or CutMix randomly per batch (50% chance each) during training.
Validation and test sets are **never augmented** — we want clean evaluation.

In [ ]:
BATCH_SIZE = 32  # we lower it
AUTOTUNE   = tf.data.AUTOTUNE

def mixup_batch(imgs, lbls):
    imgs = tf.cast(imgs, tf.float32)
    lam = tf.constant(
        np.random.beta(0.2, 0.2, BATCH_SIZE).astype(np.float32)
    )
    lam = tf.maximum(lam, 1 - lam)
    indices = tf.random.shuffle(tf.range(BATCH_SIZE))
    imgs2 = tf.gather(imgs, indices)
    lbls2 = tf.gather(lbls, indices)
    lam_img = tf.reshape(lam, [BATCH_SIZE, 1, 1, 1])
    lam_lbl = tf.reshape(lam, [BATCH_SIZE, 1])
    mixed_imgs = lam_img * imgs + (1 - lam_img) * imgs2
    lbls_oh  = tf.one_hot(tf.cast(lbls,  tf.int32), NUM_CLASSES)
    lbls2_oh = tf.one_hot(tf.cast(lbls2, tf.int32), NUM_CLASSES)
    mixed_lbls = lam_lbl * lbls_oh + (1 - lam_lbl) * lbls2_oh
    return mixed_imgs, mixed_lbls

def make_train_dataset(images, labels, batch_size=BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices((images, labels))
    ds = ds.shuffle(2000, seed=SEED)  # ul buffer
    ds = ds.batch(batch_size, drop_remainder=True)
    ds = ds.map(mixup_batch, num_parallel_calls=1)
    ds = ds.prefetch(1)
    return ds

def make_eval_dataset(images, labels, batch_size=BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices((images, labels))
    ds = ds.batch(batch_size).prefetch(1)
    return ds

train_ds = make_train_dataset(X_train, y_train)
val_ds   = make_eval_dataset(X_val,   y_val)
test_ds  = make_eval_dataset(X_test,  y_test)

print(f'Train batches : {len(train_ds)}')
print(f'Val   batches : {len(val_ds)}')
print(f'Test  batches : {len(test_ds)}')
print('Pipeline ready!')

## 6. CNN Architecture (same as baseline)

We use the **identical CNN architecture** (VGG-style, ~3.25M params).
The only difference is the augmentation strategy — this makes the comparison fair and meaningful.

**Important change:** Since MixUp/CutMix produce **soft labels** (e.g., [0.6, 0.4, 0, ...]),
we must use `CategoricalCrossentropy` instead of `SparseCategoricalCrossentropy`.

In [ ]:
def build_cnn():
    inputs = keras.Input(shape=(32, 32, 3))

    # Block 1: 32x32 → 16x16
    x = layers.Conv2D(64, 3, padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(64, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.4)(x)

    # Block 2: 16x16 → 8x8
    x = layers.Conv2D(128, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(128, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.5)(x)

    # Block 3: 8x8 → 4x4
    x = layers.Conv2D(256, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(256, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.5)(x)

    # Classifier head
    x = layers.Flatten()(x)
    x = layers.Dense(512)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(0.6)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    return keras.Model(inputs, outputs, name='CNN_MixupCutmix')

model = build_cnn()
model.summary()

## 7. Training

Same optimizer (SGD + Momentum) and schedule (CosineDecay) as the baseline.
The key difference: `CategoricalCrossentropy` to handle soft labels from MixUp/CutMix.

In [ ]:
EPOCHS = 100

lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=0.01,
    decay_steps=EPOCHS * len(train_ds)
)

optimizer = keras.optimizers.SGD(learning_rate=lr_schedule, momentum=0.9)

model.compile(
    optimizer=optimizer,
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

def encode_val(imgs, lbls):
    return imgs, tf.one_hot(tf.cast(lbls, tf.int32), NUM_CLASSES)

val_ds_oh  = val_ds.map(encode_val)
test_ds_oh = test_ds.map(encode_val)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=15,
        restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        os.path.join(model_dir, 'best_cnn_mixup_cutmix.h5'),
        monitor='val_accuracy', save_best_only=True, verbose=1
    )
]

print('Starting training...')
start = time.time()

history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds_oh,
    callbacks=callbacks,
    verbose=1
)

print(f'Training completed in {(time.time()-start)/60:.1f} minutes.')

## 8. Evaluation

In [ ]:
# Load best weights
model = keras.models.load_model(
    os.path.join(model_dir, 'best_cnn_mixup_cutmix.h5')
)

# Predict on test set
y_pred = model.predict(test_ds, verbose=0).argmax(axis=1)
y_true = y_test

test_acc = (y_pred == y_true).mean()
print(f'\n=== CNN + MixUp/CutMix Test Results ===')
print(f'Test Accuracy: {test_acc*100:.2f}%')
print(f'Baseline CNN (Cutout only): 86.89%')
print(f'Improvement: {(test_acc - 0.8689)*100:+.2f}%')

print(f'\nPer-Class Accuracy:')
print('-' * 30)
for i, name in enumerate(label_names):
    mask = y_true == i
    acc  = (y_pred[mask] == i).mean()
    print(f'  {name:<12s}: {acc*100:.1f}%')

print(f'\n{classification_report(y_true, y_pred, target_names=label_names)}')

## 9. Training Curves & Confusion Matrix

In [ ]:
epochs_range = range(1, len(history.history['loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('CNN + MixUp/CutMix — Training Curves', fontsize=13, fontweight='bold')

ax1.plot(epochs_range, history.history['loss'],     'b-', label='Train Loss')
ax1.plot(epochs_range, history.history['val_loss'], 'r-', label='Val Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Loss'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs_range, history.history['accuracy'],     'b-', label='Train Acc')
ax2.plot(epochs_range, history.history['val_accuracy'], 'r-', label='Val Acc')
ax2.axhline(y=test_acc, color='green', linestyle='--', label=f'Test: {test_acc*100:.2f}%')
ax2.axhline(y=0.8689,   color='orange', linestyle=':', label='Baseline: 86.89%')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy'); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(figure_dir, 'training_curves_mixup_cutmix.png'), dpi=150)
plt.show()

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im)

ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(label_names, rotation=45, ha='right')
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(label_names)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix — CNN + MixUp/CutMix (Test Acc: {test_acc*100:.2f}%)',
             fontweight='bold')

for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(figure_dir, 'confusion_matrix_mixup_cutmix.png'), dpi=150)
plt.show()

## 10. Final Comparison

Side-by-side comparison of baseline CNN (Cutout) vs. enhanced CNN (MixUp + CutMix).

In [ ]:
print('=' * 55)
print('AUGMENTATION COMPARISON — Custom CNN')
print('=' * 55)
print(f'{"Model":<35s} {"Test Acc":>10s}')
print('-' * 50)
print(f'{"CNN + Cutout (baseline)":<35s} {86.89:>9.2f}%')
print(f'{"CNN + MixUp/CutMix (ours)":<35s} {test_acc*100:>9.2f}%')
print('=' * 55)

models  = ['CNN\n(Cutout\nbaseline)', 'CNN\n(MixUp+CutMix\nours)']
accs    = [86.89, test_acc * 100]
colors  = ['#4C72B0', '#2ca02c']

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(models, accs, color=colors, width=0.4, edgecolor='black', linewidth=0.5)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{acc:.2f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Augmentation Impact on CNN Performance', fontsize=13, fontweight='bold')
ax.set_ylim(80, 100)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(figure_dir, 'augmentation_comparison.png'), dpi=150)
plt.show()

## Summary

| What changed | Why |
|---|---|
| Cutout → **MixUp + CutMix** | More powerful regularization: blends real image information instead of wasting it with black patches |
| `SparseCategoricalCrossentropy` → **`CategoricalCrossentropy`** | Soft labels from MixUp/CutMix require soft loss |
| Added **label smoothing (0.1)** | Extra regularization — prevents overconfidence |
| Early stopping patience **10→15** | MixUp/CutMix make training noisier, so more patience needed |


In [ ]:
# Redundant path redefinition removed to keep BASE_PATH consistent throughout.\n

In [ ]:
from sklearn.manifold import TSNE
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
import os


# 1. Load the Custom CNN Model
cnn_model = keras.models.load_model(os.path.join(model_dir, 'best_cnn_mixup_cutmix.h5'))

# 2. Build a Feature Extractor (Targeting the 512-dim Dense layer before classification)
feature_layer = None
for layer in cnn_model.layers:
    if hasattr(layer, 'units') and layer.units == 512:
        feature_layer = layer
        break

if feature_layer is None:
    # Fallback to the Dense/ReLU layer before the final classification head
    feature_layer = cnn_model.layers[-5]

cnn_feat_extractor = keras.Model(
    inputs=cnn_model.input,
    outputs=feature_layer.output
)

# 3. Select a subset of 2,000 test samples
np.random.seed(SEED)
idx_subset = np.random.choice(len(X_test), 2000, replace=False)
X_sub = X_test[idx_subset]
y_sub = y_test[idx_subset]

# Note: Custom CNN operates on native 32x32 images, no resizing required!
print("Extracting features from Custom CNN (512D)...")
features_cnn = cnn_feat_extractor.predict(X_sub, batch_size=64, verbose=1)
print(f"Feature shape: {features_cnn.shape}")

# 4. Run t-SNE (projects 512D to 2D)
print("Running t-SNE (1-2 minutes)...")
tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, n_iter=1000)
features_2d_cnn = tsne.fit_transform(features_cnn)

# 5. Plot and Save t-SNE Visualization
colors = plt.cm.tab10(np.linspace(0, 1, NUM_CLASSES))

fig, ax = plt.subplots(figsize=(12, 9))
for i, (name, color) in enumerate(zip(label_names, colors)):
    mask = y_sub == i
    ax.scatter(features_2d_cnn[mask, 0], features_2d_cnn[mask, 1],
               c=[color], label=name, alpha=0.6, s=15)

ax.set_title('t-SNE of Custom CNN Features — CIFAR-10 Test Set\n(2000 samples, 512D → 2D)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('t-SNE Component 1')
ax.set_ylabel('t-SNE Component 2')
ax.legend(loc='best', markerscale=2, fontsize=9)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(os.path.join(figure_dir, 'tsne_cnn_features.png'), dpi=150)
plt.show()
print('t-SNE visualization saved to Figure/tsne_cnn_features.png.')